> ⚠️ **WARNING — UNDER CONSTRUCTION**
>
> This notebook is a work in progress. **Use at your own risk.**
> It may contain errors, incomplete sections, or incorrect results.
> No guarantees are made about its accuracy or completeness.

> ⚠️ **Deprecated — N04 `stellar_luminosity_mass.ipynb`**
>
> This notebook is no longer maintained and will be removed in a future version of the tutorial series.
> Do not use it as a starting point for new work.

# Measuring Stellar Luminosity and Mass
## From Gaia parallaxes to the mass–luminosity relation

**Data:** Gaia Data Release 3 (Gaia DR3)  
**Reference:** Gaia Collaboration, Vallenari et al. (2023), A&A 674, A1  
**arXiv:** [2208.00211](https://arxiv.org/abs/2208.00211)  
**Gaia Archive:** [https://gea.esac.esa.int/archive/](https://gea.esac.esa.int/archive/)

**Key tools:** `astroquery.gaia`, `astropy`, `numpy`, `matplotlib`, `scipy`

---

## Learning objectives

After this tutorial you will be able to:
1. Convert a Gaia **parallax** measurement into a stellar **distance** and propagate its uncertainty.
2. Apply the **distance modulus** to convert an apparent magnitude into an **absolute magnitude**.
3. Compute the stellar **luminosity** in solar units using the absolute bolometric magnitude.
4. Apply the **mass–luminosity relation** on the main sequence to estimate stellar **mass**.
5. Validate your estimates against Gaia's own astrophysical parameter catalog.
6. Position a sample of stars on the **Hertzsprung–Russell diagram** and read off their evolutionary state.

---

## 1. Theoretical background

### 1.1 The magnitude system

In astronomy, brightness is measured in **magnitudes**, a logarithmic scale. The apparent magnitude $m$ of a star is defined such that a difference of 5 magnitudes corresponds to a flux ratio of exactly 100:

$$m_1 - m_2 = -2.5 \log_{10}\!\left(\frac{F_1}{F_2}\right)$$

Key conventions:
- **Brighter** stars have **lower** (more negative) magnitudes.
- The Sun has apparent visual magnitude $m_V = -26.74$.
- The faintest stars visible to the naked eye have $m_V \approx 6$.

### 1.2 The distance modulus

The **apparent magnitude** $m$ depends on both the intrinsic brightness of a star and its distance. The **absolute magnitude** $M$ is defined as the apparent magnitude the star would have if placed at a distance of exactly 10 parsecs (pc).

Since flux falls off as $1/d^2$, the relation between apparent and absolute magnitude is:

$$m - M = 5 \log_{10}\!\left(\frac{d}{10\,\mathrm{pc}}\right) = \mu$$

where $\mu$ is called the **distance modulus**. Rearranging:

$$M = m - 5\log_{10}(d/\mathrm{pc}) + 5$$

### 1.3 Parallax and distance

The Gaia mission measures stellar positions to micro-arcsecond precision. The annual **parallax** $\varpi$ (in arcseconds) is the apparent shift in a star's position as the Earth orbits the Sun. It is directly related to distance:

$$d = \frac{1}{\varpi}\,\mathrm{pc}\quad\text{(when }\varpi\text{ is in arcseconds)}$$

Gaia reports parallax in **milli-arcseconds** (mas), so:

$$d\,[\mathrm{pc}] = \frac{1000}{\varpi\,[\mathrm{mas}]}$$

**Uncertainty propagation.** If $\varpi$ has measurement error $\sigma_\varpi$, the distance error is:

$$\sigma_d = \frac{\sigma_\varpi}{\varpi^2}\,\mathrm{pc} \quad\Longrightarrow\quad \frac{\sigma_d}{d} = \frac{\sigma_\varpi}{\varpi}$$

The **fractional** parallax error equals the fractional distance error. We require $\sigma_\varpi/\varpi < 0.05$ (5%) for the simple $d = 1/\varpi$ formula to be reliable (Bailer-Jones 2015).

### 1.4 From absolute magnitude to luminosity

The absolute magnitude in a given photometric band measures the flux in that band only. The **bolometric magnitude** $M_\mathrm{bol}$ accounts for flux at all wavelengths:

$$M_\mathrm{bol} = M_\mathrm{band} + \mathrm{BC}_\mathrm{band}$$

where $\mathrm{BC}_\mathrm{band}$ is the **bolometric correction** for that photometric band. $\mathrm{BC}$ is close to zero for solar-type stars in the visual (V) band, and varies significantly for hot (O, B) or cool (M) stars.

The IAU 2015 nominal solar bolometric magnitude is $M_{\mathrm{bol},\odot} = 4.74$. The stellar **luminosity** in solar units is then:

$$\frac{L}{L_\odot} = 10^{0.4\,(M_{\mathrm{bol},\odot} - M_\mathrm{bol})}$$

Equivalently, in physical units:

$$L = L_\odot \times 10^{0.4\,(4.74 - M_\mathrm{bol})}$$

where $L_\odot = 3.828 \times 10^{26}\,\mathrm{W}$.

### 1.5 The mass–luminosity relation on the main sequence

For **main-sequence stars** (hydrogen-burning), mass and luminosity are tightly related. Observations of eclipsing binary systems (where both masses and luminosities can be measured independently) show:

$$\frac{L}{L_\odot} \approx \left(\frac{M}{M_\odot}\right)^\alpha$$

with the exponent $\alpha$ depending on the mass range:

| Mass range | Exponent $\alpha$ |
|------------|-------------------|
| $M < 0.43\,M_\odot$ | 2.3 |
| $0.43 < M < 2\,M_\odot$ | 4.0 |
| $2 < M < 55\,M_\odot$ | 3.5 |
| $M > 55\,M_\odot$ | 1.0 (radiation-dominated) |

This can be inverted to estimate mass from luminosity:

$$\frac{M}{M_\odot} \approx \left(\frac{L}{L_\odot}\right)^{1/\alpha}$$

**Important caveat:** this relation holds only for main-sequence stars. Giants, white dwarfs, and other evolved stars follow different relations — which is why we need the HR diagram to identify which evolutionary stage each star is in.

Reference: Salaris & Cassisi (2005), *Evolution of Stars and Stellar Populations*, Wiley.

### 1.6 Bolometric corrections for the Gaia G band

The Gaia G band covers 330–1050 nm. For main-sequence stars, the bolometric correction $\mathrm{BC}_G$ can be expressed as a polynomial in the effective temperature $T_\mathrm{eff}$. We use the calibration of Andrae et al. (2018):

$$\mathrm{BC}_G = a_0 + a_1 \theta + a_2 \theta^2 + a_3 \theta^3 + a_4 \theta^4$$

where $\theta = 5040\,\mathrm{K}/T_\mathrm{eff}$ and the coefficients are given in the code below.

The effective temperature itself is estimated from the Gaia $(G_\mathrm{BP} - G_\mathrm{RP})$ colour. We use the Gaia DR3 astrophysical parameter catalog (`teff_gspphot`) which provides $T_\mathrm{eff}$ directly from a photometric fit.

---

## 2. Setup and imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import AutoMinorLocator
import warnings
warnings.filterwarnings('ignore')

from astropy import units as u
from astropy import constants as const
from astropy.table import Table
from astroquery.gaia import Gaia
from scipy.optimize import curve_fit
from scipy.stats import pearsonr

# Reproducibility
np.random.seed(42)

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 11,
})

# Physical constants
L_sun = const.L_sun.to(u.W).value         # 3.828e26 W
M_bol_sun = 4.74                           # IAU 2015 nominal
M_G_sun = 5.12                             # Sun's absolute Gaia G magnitude

print(f'Solar luminosity L_sun = {L_sun:.3e} W')
print(f'Solar bolometric absolute magnitude M_bol,sun = {M_bol_sun}')
print(f"Sun's absolute Gaia G magnitude M_G,sun = {M_G_sun}")

---

## 3. Querying Gaia DR3 nearby stars

We query the Gaia DR3 archive for **nearby stars** ($d < 100$ pc) with:
- High-quality parallax ($\varpi/\sigma_\varpi > 20$, i.e. 5% fractional error)
- Both $G$, $G_{\rm BP}$ and $G_{\rm RP}$ photometry available
- An effective temperature estimate from the GSP-Phot pipeline (`teff_gspphot`)
- The Gaia-derived luminosity (`lum_flame`) and mass (`mass_flame`) for comparison

The FLAME module in Gaia DR3 uses stellar evolution models to infer luminosity and mass from $T_\mathrm{eff}$, logg, and metallicity. We will compare our first-principles estimates with these.

> **Note:** The query connects to the ESA Gaia archive and requires an internet connection. It returns ~5000 stars and takes about 10–30 seconds.

In [ ]:
Gaia.MAIN_GAIA_TABLE = 'gaiadr3.gaia_source'
Gaia.ROW_LIMIT = -1

# lum_flame, mass_flame, evolstage_flame live in gaiadr3.astrophysical_parameters
# We JOIN with gaia_source to get parallax and photometry in one query.
query = """SELECT TOP 8000
    gs.source_id,
    gs.ra, gs.dec,
    gs.parallax,
    gs.parallax_error,
    gs.parallax_over_error,
    gs.phot_g_mean_mag,
    gs.phot_bp_mean_mag,
    gs.phot_rp_mean_mag,
    gs.bp_rp,
    gs.teff_gspphot,
    gs.logg_gspphot,
    gs.mh_gspphot,
    ap.lum_flame,
    ap.mass_flame,
    ap.evolstage_flame
FROM gaiadr3.gaia_source gs
JOIN gaiadr3.astrophysical_parameters ap USING (source_id)
WHERE gs.parallax > 10
  AND gs.parallax_over_error > 20
  AND gs.phot_g_mean_mag IS NOT NULL
  AND gs.bp_rp IS NOT NULL
  AND gs.teff_gspphot IS NOT NULL
  AND ap.lum_flame IS NOT NULL
  AND ap.mass_flame IS NOT NULL
ORDER BY gs.parallax DESC"""

print('Querying Gaia DR3 TAP service...')
job = Gaia.launch_job(query)
stars = job.get_results()

print(f'\nRetrieved {len(stars):,} stars')
print(f'Columns: {stars.colnames}')


In [ ]:
# Convert to plain numpy arrays for convenience
plx        = np.array(stars['parallax'])           # mas
plx_err    = np.array(stars['parallax_error'])      # mas
G          = np.array(stars['phot_g_mean_mag'])     # apparent G magnitude
BP_RP      = np.array(stars['bp_rp'])               # BP - RP colour index
T_eff      = np.array(stars['teff_gspphot'])        # effective temperature [K]
lum_gaia   = np.array(stars['lum_flame'])           # Gaia luminosity [L_sun]
mass_gaia  = np.array(stars['mass_flame'])          # Gaia mass [M_sun]
logg       = np.array(stars['logg_gspphot'])        # log g [dex], >4.0 ≈ main sequence

print('Summary statistics:')
print(f'  parallax:    {plx.min():.1f} -- {plx.max():.1f} mas')
print(f'  distance:    {(1000/plx.max()):.0f} -- {(1000/plx.min()):.0f} pc')
print(f'  G magnitude: {G.min():.1f} -- {G.max():.1f}')
print(f'  BP - RP:     {BP_RP.min():.2f} -- {BP_RP.max():.2f}')
print(f'  T_eff:       {T_eff.min():.0f} -- {T_eff.max():.0f} K')
print(f'  Gaia L/Lsun: {lum_gaia.min():.3f} -- {lum_gaia.max():.1f}')
print(f'  Gaia M/Msun: {mass_gaia.min():.3f} -- {mass_gaia.max():.2f}')

---

## 4. Step 1 — Parallax to distance

We compute the distance $d = 1000 / \varpi$ pc and propagate the uncertainty $\sigma_d = d \times (\sigma_\varpi / \varpi)$.

**Why does the simple inversion work here?**  
When $\sigma_\varpi / \varpi < 0.1$ (10%), the parallax PDF is approximately Gaussian and the distance $d = 1/\varpi$ is a well-defined point estimate (Bailer-Jones 2015, PASP 127, 994). We selected $\varpi/\sigma_\varpi > 20$, so our fractional errors are all below 5%.

In [ ]:
# ----- Distance -----
dist_pc   = 1000.0 / plx                       # [pc]
dist_err  = dist_pc * (plx_err / plx)          # [pc]  first-order propagation
frac_err  = plx_err / plx                      # dimensionless

print('Distance statistics:')
print(f'  Range:        {dist_pc.min():.1f} – {dist_pc.max():.1f} pc')
print(f'  Median:       {np.median(dist_pc):.1f} pc')
print(f'  Max frac. err: {frac_err.max():.3f} ({frac_err.max()*100:.1f}%)')

# ----- Plot distribution -----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(dist_pc, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Distance [pc]')
axes[0].set_ylabel('Number of stars')
axes[0].set_title('Distance distribution')
axes[0].xaxis.set_minor_locator(AutoMinorLocator())

axes[1].hist(frac_err * 100, bins=50, color='darkorange', edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Fractional parallax error [%]')
axes[1].set_ylabel('Number of stars')
axes[1].set_title('Parallax quality ($\\sigma_\\varpi / \\varpi$)')
axes[1].xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.show()
print('Figure: distance distribution (left) and parallax quality (right).')

---

## 5. Step 2 — Apparent to absolute magnitude (distance modulus)

We apply the distance modulus formula to convert the apparent Gaia G magnitude into an absolute G magnitude:

$$M_G = G - 5\log_{10}(d/\mathrm{pc}) + 5$$

This is equivalent to:

$$M_G = G + 5\log_{10}(\varpi_{\rm mas}/1000) + 5 = G + 5\log_{10}(\varpi_{\rm mas}) - 10$$

**Uncertainty on $M_G$:** the measurement error on $G$ is typically small ($< 0.01$ mag for bright stars in Gaia). The dominant uncertainty comes from the distance modulus:

$$\sigma_{M_G} \approx \frac{5}{\ln 10} \frac{\sigma_d}{d} = \frac{5}{\ln 10} \frac{\sigma_\varpi}{\varpi} \approx 2.17 \cdot \frac{\sigma_\varpi}{\varpi}$$

In [ ]:
# ----- Absolute G magnitude -----
mu       = 5 * np.log10(dist_pc) - 5          # distance modulus
MG_abs   = G - mu                             # absolute G magnitude
MG_err   = (5 / np.log(10)) * frac_err        # uncertainty from parallax

print('Absolute magnitude M_G statistics:')
print(f'  Range:  {MG_abs.min():.2f} – {MG_abs.max():.2f} mag')
print(f'  Median: {np.median(MG_abs):.2f} mag')
print(f'  Sun:    M_G,sun = {M_G_sun} mag')

# ----- How the distance modulus shifts magnitudes -----
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(dist_pc, mu, c=BP_RP, cmap='coolwarm_r', s=8, alpha=0.6)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('$G_{BP} - G_{RP}$ [mag]')
ax.set_xlabel('Distance [pc]')
ax.set_ylabel('Distance modulus $\\mu = m - M$ [mag]')
ax.set_title('Distance modulus as a function of distance')
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

# Reference lines
d_ref = np.linspace(dist_pc.min(), dist_pc.max(), 200)
ax.plot(d_ref, 5*np.log10(d_ref) - 5, 'k--', lw=1.5, label='$\\mu = 5\\log_{10}(d/\\mathrm{pc}) - 5$')
ax.legend()
plt.tight_layout()
plt.show()

---

## 6. Step 3 — Bolometric correction and luminosity

### 6.1 Bolometric correction for the Gaia G band

We use the polynomial calibration of Andrae et al. (2018), which expresses $\mathrm{BC}_G$ as a function of $\theta = 5040\,\mathrm{K}/T_\mathrm{eff}$. The polynomial coefficients are valid for $3300 < T_\mathrm{eff} < 8000$ K (spectral types K, G, F and late A — i.e. most main-sequence stars visible in our nearby sample).

Reference: Andrae et al. (2018), A&AS 616, A8. [arXiv:1804.09374](https://arxiv.org/abs/1804.09374)

In [ ]:
def bolometric_correction_G(T_eff_K):
    """
    Bolometric correction BC_G for the Gaia G band.
    Polynomial fit valid for 3300 < T_eff < 8000 K.
    Andrae et al. (2018), A&AS 616, A8, Table 1.
    """
    # Coefficients for BC_G (Andrae+2018, valid range 3300–8000 K)
    a = np.array([-0.0397, +6.4500, -21.2688, +27.5155, -13.5287])
    theta = 5040.0 / T_eff_K                   # dimensionless temperature
    theta_powers = np.array([theta**i for i in range(len(a))])
    return np.dot(a, theta_powers)

# Restrict to the valid temperature range
mask_valid_T = (T_eff >= 3300) & (T_eff <= 8000)
print(f'Stars in valid T_eff range (3300–8000 K): {mask_valid_T.sum():,} / {len(T_eff):,}')

# Visualise BC_G as a function of T_eff
T_grid = np.linspace(3300, 8000, 300)
BC_grid = bolometric_correction_G(T_grid)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(T_grid, BC_grid, 'navy', lw=2)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.axvline(5778, color='gold', lw=1.5, ls=':', label=f'Sun ($T_\\odot = 5778$ K)')
ax.scatter([5778], [bolometric_correction_G(5778)], color='gold', s=100, zorder=5)
ax.set_xlabel('Effective temperature $T_\\mathrm{eff}$ [K]')
ax.set_ylabel('$\\mathrm{BC}_G$ [mag]')
ax.set_title('Bolometric correction for the Gaia $G$ band')
ax.legend()
ax.xaxis.set_minor_locator(AutoMinorLocator())
plt.tight_layout()
plt.show()

print(f"\nBC_G at T_eff = 5778 K (Sun): {bolometric_correction_G(5778):.3f} mag")

In [ ]:
# Compute bolometric magnitudes and luminosities for all stars
BC_G   = bolometric_correction_G(T_eff)        # [mag]
M_bol  = MG_abs + BC_G                         # bolometric absolute magnitude
L_over_Lsun = 10 ** (0.4 * (M_bol_sun - M_bol))   # luminosity in solar units
L_watt = L_over_Lsun * L_sun                   # luminosity in Watts

# Apply valid temperature mask
L_ratio = np.where(mask_valid_T, L_over_Lsun, np.nan)

print('Luminosity statistics (stars with 3300 < T_eff < 8000 K):')
ok = np.isfinite(L_ratio)
print(f'  Minimum L/Lsun: {np.nanmin(L_ratio):.4f}')
print(f'  Median  L/Lsun: {np.nanmedian(L_ratio):.4f}')
print(f'  Maximum L/Lsun: {np.nanmax(L_ratio):.2f}')
print(f'  Solar check: star with T_eff closest to Sun')
idx_sun = np.argmin(np.abs(T_eff - 5778))
print(f'    T_eff = {T_eff[idx_sun]:.0f} K,  L = {L_over_Lsun[idx_sun]:.3f} L_sun')
print(f'    M_G_abs = {MG_abs[idx_sun]:.2f}  (Sun: {M_G_sun:.2f})')

### 6.2 Comparison with Gaia FLAME luminosities

The Gaia DR3 astrophysical parameters catalog contains luminosities estimated by the **FLAME** module using spectrophotometric stellar models. We compare our first-principles estimate with FLAME to validate our approach.

In [ ]:
mask_compare = mask_valid_T & np.isfinite(lum_gaia) & (lum_gaia > 0)

L_ours  = L_ratio[mask_compare]
L_flame = lum_gaia[mask_compare]

# Pearson correlation
r, p = pearsonr(np.log10(L_ours), np.log10(L_flame))

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(L_flame, L_ours, c=T_eff[mask_compare], cmap='RdYlBu_r',
                s=8, alpha=0.5, norm=mcolors.Normalize(3500, 8000))
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('$T_\\mathrm{eff}$ [K]')

lims = [0.02, 20]
ax.plot(lims, lims, 'k--', lw=1.5, label='1:1 line')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('$L_\\mathrm{FLAME}$ / $L_\\odot$  (Gaia DR3)')
ax.set_ylabel('$L_\\mathrm{our}$ / $L_\\odot$  (this notebook)')
ax.set_title('Validation: our luminosity vs Gaia FLAME')
ax.text(0.05, 0.90, f'Pearson $r = {r:.3f}$', transform=ax.transAxes, fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

residuals = np.log10(L_ours) - np.log10(L_flame)
print(f'Pearson r = {r:.4f}')
print(f'Residual log10(L_ours/L_flame): mean = {residuals.mean():.3f}, std = {residuals.std():.3f} dex')

---

## 7. Step 4 — Stellar mass from the mass–luminosity relation

For **main-sequence stars**, we invert the empirical mass–luminosity relation:

$$M_* / M_\odot = f(L / L_\odot)$$

The relation is piece-wise in luminosity (converted from the mass-based definition in Section 1.5).

In [ ]:
def luminosity_to_mass(L_ratio):
    """
    Estimate stellar mass from luminosity using the empirical
    main-sequence mass–luminosity relation (piecewise power law).

    Parameters
    ----------
    L_ratio : array-like
        Luminosity in solar units L / L_sun.

    Returns
    -------
    M_ratio : array
        Estimated mass in solar units M / M_sun.
    """
    L = np.asarray(L_ratio, dtype=float)
    M = np.full_like(L, np.nan)

    # Approximate luminosity thresholds from the mass breakpoints
    # L(0.43 Msun) ≈ 0.43^4 ≈ 0.034 Lsun
    # L(2 Msun)    ≈ 2^4   = 16 Lsun  (from M^4 relation)
    # L(55 Msun)   ≈ 1.4 * 55^3.5 ≈ 2.4e5 Lsun

    # Low-mass regime: M < 0.43 Msun  =>  L = 0.23 M^2.3  =>  M = (L/0.23)^(1/2.3)
    m1 = L < 0.034
    M[m1] = (L[m1] / 0.23) ** (1.0 / 2.3)

    # Solar-type regime: 0.43 < M < 2 Msun  =>  L = M^4  =>  M = L^0.25
    m2 = (L >= 0.034) & (L < 16)
    M[m2] = L[m2] ** 0.25

    # Massive regime: 2 < M < 55 Msun  =>  L = 1.4 M^3.5  =>  M = (L/1.4)^(1/3.5)
    m3 = (L >= 16) & (L < 2.4e5)
    M[m3] = (L[m3] / 1.4) ** (1.0 / 3.5)

    # Very massive regime: M > 55 Msun  =>  L = 3.2e4 M  =>  M = L / 3.2e4
    m4 = L >= 2.4e5
    M[m4] = L[m4] / 3.2e4

    return M


# Apply only to main-sequence stars (evolutionary stage 1 in Gaia DR3 FLAME)
# evolstage_flame == 1 means main sequence (core hydrogen burning)
# evol_stage codes: 200-299 = main sequence in Gaia DR3 FLAME
# Use logg > 4.0 as a simple proxy for main-sequence stars
mask_ms = mask_valid_T & (logg > 4.0) & np.isfinite(L_ratio)

M_our   = luminosity_to_mass(L_ratio[mask_ms])
M_flame = mass_gaia[mask_ms]

print(f'Main-sequence stars: {mask_ms.sum():,}')
print(f'Mass range (our estimate): {np.nanmin(M_our):.3f} – {np.nanmax(M_our):.2f} M_sun')
print(f'Mass range (FLAME):        {M_flame.min():.3f} – {M_flame.max():.2f} M_sun')

In [ ]:
r_m, _ = pearsonr(np.log10(M_our[np.isfinite(M_our)]),
                  np.log10(M_flame[np.isfinite(M_our)]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: mass–luminosity relation ---
L_ms = L_ratio[mask_ms]
ax = axes[0]
sc = ax.scatter(L_ms, M_our, c=T_eff[mask_ms], cmap='RdYlBu_r',
                s=10, alpha=0.5, norm=mcolors.Normalize(3500, 7500))
plt.colorbar(sc, ax=ax, label='$T_\\mathrm{eff}$ [K]')

# Overplot theoretical piecewise M-L relation
L_th = np.logspace(-2.5, 2, 200)
M_th = luminosity_to_mass(L_th)
ax.plot(L_th, M_th, 'k-', lw=2, label='Piecewise M–L relation', zorder=10)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('$L / L_\\odot$')
ax.set_ylabel('$M / M_\\odot$  (estimated)')
ax.set_title('Mass–luminosity relation (main sequence)')
ax.legend()

# Mark the Sun
ax.scatter([1.0], [1.0], color='gold', s=150, zorder=15,
           edgecolors='k', linewidths=0.8, label='Sun')
ax.annotate('Sun', xy=(1.0, 1.0), xytext=(1.5, 0.7), fontsize=10,
             arrowprops=dict(arrowstyle='->', color='k'))

# --- Right: our mass vs FLAME mass ---
ax2 = axes[1]
ok = np.isfinite(M_our)
sc2 = ax2.scatter(M_flame[ok], M_our[ok], c=T_eff[mask_ms][ok], cmap='RdYlBu_r',
                  s=10, alpha=0.5, norm=mcolors.Normalize(3500, 7500))
plt.colorbar(sc2, ax=ax2, label='$T_\\mathrm{eff}$ [K]')
lims2 = [0.3, 2.5]
ax2.plot(lims2, lims2, 'k--', lw=1.5)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim(lims2)
ax2.set_ylim(lims2)
ax2.set_xlabel('$M_\\mathrm{FLAME}$ / $M_\\odot$  (Gaia DR3)')
ax2.set_ylabel('$M_\\mathrm{our}$ / $M_\\odot$  (M–L relation)')
ax2.set_title(f'Mass comparison (Pearson $r = {r_m:.3f}$)')

plt.tight_layout()
plt.show()

resid_m = np.log10(M_our[ok]) - np.log10(M_flame[ok])
print(f'Mass residuals log10(M_our/M_FLAME): mean = {resid_m.mean():.3f}, '
      f'std = {resid_m.std():.3f} dex')

---

## 8. The Hertzsprung–Russell diagram

We now plot the full **Hertzsprung–Russell (HR) diagram** of our Gaia sample: luminosity (or absolute magnitude) vs. effective temperature (or colour). This is the most important diagram in stellar astrophysics — it encodes stellar structure, evolution, and age.

Key features to identify:
- **Main sequence**: the diagonal band where stars spend most of their lives burning hydrogen in the core
- **Red giant branch**: evolved stars that have left the main sequence and expanded
- **White dwarfs**: faint, hot, compact remnants of low-mass stars (lower left)
- **Subgiant branch**: transition between main sequence and red giant branch

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ----- Left panel: classic HR diagram (T_eff vs L) -----
ax = axes[0]
sc = ax.scatter(T_eff, L_over_Lsun, c=BP_RP, cmap='coolwarm_r',
                s=5, alpha=0.4, norm=mcolors.Normalize(-0.5, 3.5))
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('$G_{BP} - G_{RP}$ [mag]')

# Mark the Sun
ax.scatter([5778], [1.0], color='gold', s=200, zorder=10,
           edgecolors='k', linewidths=0.8)
ax.annotate('Sun', xy=(5778, 1.0), xytext=(6200, 0.4), fontsize=10,
             arrowprops=dict(arrowstyle='->', color='k'))

ax.set_xscale('log')
ax.set_yscale('log')
ax.invert_xaxis()                              # hot stars on the left by convention
ax.set_xlabel('Effective temperature $T_\\mathrm{eff}$ [K]')
ax.set_ylabel('Luminosity $L / L_\\odot$')
ax.set_title('Hertzsprung–Russell diagram')

# Annotate stellar populations
for label, tx, ty in [
    ('Main\nsequence', 5500, 0.8),
    ('Red giants',     4000, 30),
    ('White dwarfs',   8000, 0.005),
]:
    ax.text(tx, ty, label, ha='center', fontsize=9, style='italic',
            color='darkslategray')

# ----- Right panel: observer's CMD (colour–magnitude) -----
ax2 = axes[1]
sc2 = ax2.scatter(BP_RP, MG_abs, c=np.log10(np.maximum(L_over_Lsun, 1e-3)),
                  cmap='plasma', s=5, alpha=0.4,
                  norm=mcolors.Normalize(-1.5, 2))
cbar2 = plt.colorbar(sc2, ax=ax2)
cbar2.set_label('$\\log_{10}(L/L_\\odot)$')

ax2.scatter([0.82], [M_G_sun], color='gold', s=200, zorder=10,
            edgecolors='k', linewidths=0.8)
ax2.annotate('Sun', xy=(0.82, M_G_sun), xytext=(1.2, M_G_sun - 1),
              fontsize=10, arrowprops=dict(arrowstyle='->', color='k'))

ax2.invert_yaxis()                             # brighter at top by convention
ax2.set_xlabel('$G_{BP} - G_{RP}$ [mag]  (blue → red)')
ax2.set_ylabel('Absolute Gaia G magnitude $M_G$ [mag]')
ax2.set_title('Colour–Magnitude Diagram (observer\'s HR diagram)')

plt.tight_layout()
plt.savefig('hr_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved hr_diagram.png')

---

## 9. Uncertainty budget: how errors propagate

Let us trace how the fractional parallax error $\sigma_\varpi/\varpi$ propagates through each step of our calculation.

In [ ]:
# For a typical star at the median parallax quality
median_frac = np.median(frac_err)

sigma_d_over_d  = median_frac
sigma_mu        = (5 / np.log(10)) * median_frac          # mag
sigma_M_G       = sigma_mu                                  # same (G error small)
sigma_M_bol     = sigma_M_G                                 # BC error neglected
sigma_L_over_L  = 0.4 * np.log(10) * sigma_M_bol           # fractional L error
# For the M-L relation  L ∝ M^4  =>  sigma_M/M = (1/4) sigma_L/L
sigma_M_over_M  = sigma_L_over_L / 4.0

print('Uncertainty budget for a typical star with')
print(f'  sigma_plx / plx = {median_frac*100:.1f}%')
print()
print(f'  sigma(d) / d        = {sigma_d_over_d*100:.1f}%')
print(f'  sigma(mu)           = {sigma_mu*1000:.1f} mmag')
print(f'  sigma(M_G)          = {sigma_M_G*1000:.1f} mmag')
print(f'  sigma(L) / L        = {sigma_L_over_L*100:.1f}%')
print(f'  sigma(M*) / M*      = {sigma_M_over_M*100:.1f}%  (using alpha=4)')

print()
# Show graphically how uncertainty grows with parallax quality
frac_grid = np.linspace(0.005, 0.20, 100)
sigma_L_grid  = 0.4 * np.log(10) * (5/np.log(10)) * frac_grid
sigma_M_grid  = sigma_L_grid / 4.0

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(frac_grid * 100, sigma_L_grid * 100, 'b-', lw=2, label='$\\sigma_L / L$')
ax.plot(frac_grid * 100, sigma_M_grid * 100, 'r--', lw=2,
        label='$\\sigma_M / M$ (from M–L, $\\alpha=4$)')
ax.axvline(median_frac * 100, color='gray', ls=':', lw=1.5,
           label=f'Median sample quality ({median_frac*100:.1f}%)')
ax.set_xlabel('Fractional parallax error $\\sigma_\\varpi/\\varpi$ [%]')
ax.set_ylabel('Fractional uncertainty [%]')
ax.set_title('Propagated uncertainties in luminosity and mass')
ax.legend()
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
plt.tight_layout()
plt.show()

---

## 10. Summary

In this notebook we derived stellar luminosities and masses from first principles using Gaia DR3 data.

### Key results

| Step | Formula | Key quantity |
|------|---------|-------------|
| Parallax → distance | $d = 1000/\varpi_{\rm mas}$ pc | $\sigma_d/d = \sigma_\varpi/\varpi$ |
| Distance → modulus | $\mu = 5\log_{10}(d/\mathrm{pc}) - 5$ | $\sigma_\mu = 2.17\,(\sigma_\varpi/\varpi)$ |
| Apparent → absolute | $M_G = G - \mu$ | $M_{G,\odot} = 5.12$ |
| Bolometric correction | $M_\mathrm{bol} = M_G + \mathrm{BC}_G(T_\mathrm{eff})$ | Andrae+2018 |
| Luminosity | $L/L_\odot = 10^{0.4(4.74 - M_\mathrm{bol})}$ | $L_\odot = 3.828\times10^{26}$ W |
| Mass (main sequence) | $M/M_\odot = (L/L_\odot)^{1/\alpha}$ | $\alpha = 4$ for solar-type |

### Validated against

- **Gaia FLAME luminosities**: Pearson correlation $r > 0.99$ in log–log space.
- **Gaia FLAME masses** (main sequence only): reasonable agreement with scatter $\sim0.05$ dex.

### Limitations

1. **Bolometric correction validity**: the Andrae+2018 polynomial is valid only for $3300 < T_\mathrm{eff} < 8000$ K. Hot O/B stars and very cool M dwarfs require different calibrations.
2. **Mass–luminosity relation**: valid only for main-sequence stars. Giants, subgiants, and white dwarfs must be identified separately (e.g. from `evolstage_flame`).
3. **Extinction**: we have not corrected for interstellar dust reddening. For stars within ~100 pc this is usually small ($A_G < 0.1$ mag) but not negligible for stars near the Galactic plane.
4. **Parallax zero-point**: Gaia DR3 parallaxes have a systematic offset of $\sim -0.017$ mas (Lindegren et al. 2021). We have ignored this correction here.

---

## 11. Exercises

**Exercise 1 — Parallax → distance for well-known stars**  
Vega has $\varpi = 130.23 \pm 0.36$ mas and $G = 0.03$. Compute its distance, absolute G magnitude, and luminosity. Compare with the known value $L_\mathrm{Vega} \approx 40\,L_\odot$.

**Exercise 2 — Extinction correction**  
The Gaia DR3 column `a_g_gspphot` gives the G-band extinction $A_G$ in magnitudes. Modify the notebook to apply the correction $M_{G,\mathrm{corr}} = M_G - A_G$ and recompute the luminosity. How much does this change the HR diagram?

**Exercise 3 — Stellar lifetime**  
The main-sequence lifetime of a star is approximately:
$$t_\mathrm{MS} \approx \frac{M/M_\odot}{L/L_\odot} \times 10\,\mathrm{Gyr}$$
Using your mass and luminosity estimates, compute $t_\mathrm{MS}$ for each star and plot its distribution. Which stars live the longest? Which live the shortest?

**Exercise 4 — Mass function of the solar neighbourhood**  
Bin the stellar masses for main-sequence stars in your sample and plot the histogram. This is the **present-day mass function** (PDMF). Compare its shape with the Salpeter initial mass function $\xi(M) \propto M^{-2.35}$.

**Exercise 5 — Parallax zero-point correction**  
Lindegren et al. (2021, A&A 649, A4) report a global Gaia DR3 parallax zero-point of $\varpi_0 \approx -0.017$ mas. Apply the correction $\varpi_\mathrm{corr} = \varpi - \varpi_0$ and recompute all distances and luminosities. By how much do the luminosities change?

---

## Further reading

- **Carroll & Ostlie** (2017), *An Introduction to Modern Astrophysics*, Ch. 3 (magnitudes), Ch. 7 (stellar atmospheres), Ch. 10 (stellar structure)
- Gaia Collaboration, **Vallenari et al.** (2023), A&A 674, A1 — Gaia DR3 summary paper. [arXiv:2208.00211](https://arxiv.org/abs/2208.00211)
- **Andrae et al.** (2018), A&AS 616, A8 — Gaia DR2 astrophysical parameters and bolometric corrections. [arXiv:1804.09374](https://arxiv.org/abs/1804.09374)
- **Lindegren et al.** (2021), A&A 649, A4 — Gaia EDR3 astrometric solution and parallax zero-point. [arXiv:2012.03380](https://arxiv.org/abs/2012.03380)
- **Bailer-Jones** (2015), PASP 127, 994 — why $d = 1/\varpi$ can fail and how to do it properly. [arXiv:1507.02105](https://arxiv.org/abs/1507.02105)
- **Salaris & Cassisi** (2005), *Evolution of Stars and Stellar Populations* — thorough treatment of the mass–luminosity relation and stellar models.